# Solutions · 09 · Observers and the separation principle

Worked solutions to the exercises of [notebook 09](../notebooks/09_observers_and_the_separation_principle.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, statespace as sp, response as rs, frequency as fr, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Place the observer poles at 2, 5 and 20 times the controller's slowest pole and compare the step responses, the sensitivity to measurement noise (simulate with noise added to y) and the loop margins. Where is the sweet spot?

In [2]:
mot = models.dc_motor(position=True)["system"]
K = sp.place(mot.A, mot.B, [-8 + 6j, -8 - 6j, -20.0])
slowest = 8.0
rng = np.random.default_rng(0)
rows = []
for factor in (2.0, 5.0, 20.0):
    L = sp.observer_gain(mot.A, mot.C, -factor * slowest * np.array([1.0, 1.1, 1.2]))
    cl = sp.closed_loop_with_observer(mot, K, L)
    t, y = rs.step(cl, 1.5)
    comp = sy.ss(mot.A - mot.B @ K - L @ mot.C, L, K, np.zeros((1, 1)))
    L_in = sy.series(mot.to_tf(), comp.to_tf())
    m = fr.margins(L_in)
    # noise: the estimate's response to measurement noise through the observer
    obs = sp.observer(mot, L)
    tn = np.linspace(0, 2, 2001)
    _, xh, _ = rs.lsim(obs, np.column_stack([np.zeros(2001), 0.01 * rng.normal(size=2001)]), tn)
    rows.append((factor, rs.step_info(t, y, final=1.0)["settling_time"], m["gain_margin"], m["phase_margin"], np.std(xh[:, 1])))
print(pd.DataFrame(rows, columns=["observer speed / controller", "settling (s)", "gain margin", "phase margin (deg)", "speed-estimate noise (rad/s)"]).to_string(index=False, float_format="%.3f"))

 observer speed / controller  settling (s)  gain margin  phase margin (deg)  speed-estimate noise (rad/s)
                       2.000         0.447        3.024              43.107                         0.015
                       5.000         0.447        3.848              46.189                         0.092
                      20.000         0.447        8.682              61.152                         0.873


The step response does not change at all - the separation principle promises that - but two things do. The
noise in the speed estimate grows sixtyfold between the slowest and the fastest observer, and the loop's
margins *improve* with a faster observer: for this well-modelled motor the observer-based loop approaches
the full-state-feedback loop as the observer gets fast (loop transfer recovery). The sweet spot is where the
margins are acceptable and the noise still tolerable, a few times the controller's speed. The project's
servo, with an unmodelled resonance, turns the margin trend the other way round - which is why the choice
has to be made on the loop that is actually closed.

## Exercise 2

Design a reduced-order observer for the DC motor that estimates only the current from the measured speed (the speed is measured, so only one state needs estimating). Compare its order and convergence with the full-order observer.

In [3]:
mot = models.dc_motor()["system"]            # states [current, speed]; speed measured
A, B, C = mot.A, mot.B, mot.C
# reduced-order observer for the current (state 1) from the measured speed (state 2):
# with x = [i, w]: i' = A11 i + A12 w + B1 u ; w' = A21 i + A22 w + B2 u  ->  the second equation is a measurement of i: w' - A22 w - B2 u = A21 i
A11, A12, A21, A22 = A[0, 0], A[0, 1], A[1, 0], A[1, 1]; B1, B2 = B[0, 0], B[1, 0]
pole = -60.0
Lr = (A11 - pole) / A21                              # observer gain: error dynamics e' = (A11 - Lr A21) e
# z = i_hat - Lr w obeys z' = (A11 - Lr A21) z + (A12 - Lr A22 + (A11 - Lr A21) Lr) w + (B1 - Lr B2) u
t = np.linspace(0, 1, 1001); u = 12.0 * np.ones_like(t)
_, y, x = rs.lsim(mot, u, t, x0=[0.5, 5.0])
z = 0.0; ihat = np.zeros_like(t); dt = t[1] - t[0]
Fz = A11 - Lr * A21; Gz = A12 - Lr * A22 + Fz * Lr; Hz = B1 - Lr * B2
for k in range(len(t)):
    ihat[k] = z + Lr * y[k]
    z = z + dt * (Fz * z + Gz * y[k] + Hz * u[k])
err = ihat - x[:, 0]
print(f"reduced-order observer (1 state, pole {pole}): current error {err[0]:+.3f} A at t = 0, {err[100]:+.1e} at 0.1 s, {err[-1]:+.1e} at 1 s")
print("a full-order observer would have 2 states and two poles to place; the reduced one estimates only what is not measured")

reduced-order observer (1 state, pole -60.0): current error +289.500 A at t = 0, +7.0e-01 at 0.1 s, -3.2e-04 at 1 s
a full-order observer would have 2 states and two poles to place; the reduced one estimates only what is not measured


When part of the state is measured directly, only the rest needs estimating: the reduced-order observer has
as many states as unmeasured variables (one here, the current), uses the measured speed's derivative
implicitly through the change of variable z = î − L_r ω, and converges at its single pole (from a wild initial
value - the gain L_r is large - to 10⁻⁴ A within a second). It is smaller and
avoids filtering a measurement that is already clean; the full-order observer is simpler to derive and
smooths the measurement as a by-product.

## Exercise 3

**Implement it yourself:** build the observer-based compensator from (A, B, C, K, L) as a transfer function from y to u and close the loop with the plant in the transfer-function domain; confirm that the closed-loop poles are the union of the two designed sets.

In [4]:
cp = models.cart_pendulum()["system"]
A, B, C = cp.A, cp.B, cp.C
K = sp.place(A, B, [-2 + 2j, -2 - 2j, -5.0, -6.0])
L = sp.observer_gain(A, C, [-10.0, -11.0, -12.0, -13.0])
comp = sy.ss(A - B @ K - L @ C, L, -K, np.zeros((1, 2)))        # y (2 outputs) -> u
# closed loop by state-space interconnection: u = comp(y), y = plant(u)
Acl = np.block([[A, B @ comp.C], [comp.B @ C, comp.A]])
print("closed-loop eigenvalues:", np.round(np.sort_complex(np.linalg.eigvals(Acl)), 4))
print("= union of the designed sets:", np.allclose(np.sort_complex(np.linalg.eigvals(Acl)), np.sort_complex(np.r_[np.linalg.eigvals(A - B @ K), np.linalg.eigvals(A - L @ C)])))

closed-loop eigenvalues: [-13.+0.j -12.+0.j -11.+0.j -10.+0.j  -6.+0.j  -5.+0.j  -2.-2.j  -2.+2.j]
= union of the designed sets: True


The compensator from y to u is the observer with u = −Kx̂ substituted; closing it around the plant gives an
8th-order system whose eigenvalues are the four controller poles and the four observer poles - the
separation principle holds in the compensator form too, as it must, since the two forms differ only by a
change of variables (x, x̂) → (x, x − x̂).